# Exercise 3: Backpropagation — The Engine of Deep Learning
## Build Your Own Autograd Engine

---

### Learning Objectives
- Understand the computational graph and how gradients flow backward through it
- Build a **miniature autograd engine** (like micrograd) from scratch
- Visualize gradient flow and understand the vanishing/exploding gradient problem
- See how Batch Normalization and skip connections (ResNet) solve these problems

### Why This Matters
Every deep learning framework (PyTorch, TensorFlow, JAX) uses automatic differentiation under the hood. Understanding how it works makes you a better engineer — you'll know when gradients explode, why training stalls, and how to debug it.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import torch
import torch.nn as nn

plt.style.use('seaborn-v0_8-darkgrid')
print('Ready!')

## Part 1: Build a Miniature Autograd Engine

We'll build a `Value` class that tracks:
1. The **data** (forward pass result)
2. The **gradient** (backprop result)
3. A `_backward` function that knows how to propagate gradients through this specific operation

This is essentially how PyTorch's `autograd` works, just simplified.

In [ ]:
class Value:
    """A scalar value that supports automatic differentiation."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            # Gradient of a + b = 1 for both a and b
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            # Gradient of a * b: d/da = b, d/db = a
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = np.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(np.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(np.log(self.data + 1e-15), (self,), 'log')
        def _backward():
            self.grad += (1.0 / (self.data + 1e-15)) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        """Run backpropagation using topological sort."""
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * (other ** -1)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __repr__(self): return f'Value({self.data:.4f}, grad={self.grad:.4f})'

print('Value class defined!')

In [ ]:
# === Demo: trace gradients through a simple expression ===
# Expression: L = (x * w + b)^2

x = Value(2.0, label='x')
w = Value(-3.0, label='w')
b = Value(6.8813735870195432, label='b')

xw = x * w;    xw.label = 'x*w'
n  = xw + b;   n.label  = 'n'
o  = n.tanh(); o.label  = 'o (loss)'

o.backward()

print('=== Forward Pass ===')
print(f'x*w = {xw}')
print(f'n   = {n}')
print(f'o   = {o}')

print('\n=== Backward Pass (gradients) ===')
print(f'dL/dx = {x.grad:.4f}  (numerically: {(np.tanh((-3*2.01+6.88)) - np.tanh((-3*2+6.88))) / 0.01:.4f})')
print(f'dL/dw = {w.grad:.4f}  (numerically: {(np.tanh((-3.01*2+6.88)) - np.tanh((-3*2+6.88))) / 0.01:.4f})')
print(f'dL/db = {b.grad:.4f}')

# Verify with PyTorch
print('\n=== Verification with PyTorch ===')
xt = torch.tensor([2.0], requires_grad=True)
wt = torch.tensor([-3.0], requires_grad=True)
bt = torch.tensor([6.8813735870195432], requires_grad=True)
ot = (xt * wt + bt).tanh()
ot.backward()
print(f'dL/dx = {xt.grad.item():.4f} ✓' if abs(xt.grad.item() - x.grad) < 1e-5 else f'MISMATCH!')
print(f'dL/dw = {wt.grad.item():.4f} ✓' if abs(wt.grad.item() - w.grad) < 1e-5 else f'MISMATCH!')

## Part 2: Visualizing Gradient Flow in Deep Networks

How does gradient magnitude change as it flows from the output layer back to the input? With sigmoid activation, it shrinks exponentially — the **vanishing gradient problem**.

In [ ]:
def measure_gradient_flow(activation, num_layers=20, hidden_size=256):
    """
    Build a deep network, do one forward/backward pass,
    and record the gradient norm at each layer.
    """
    layers = []
    for i in range(num_layers):
        in_f = 2 if i == 0 else hidden_size
        layers.append(nn.Linear(in_f, hidden_size))
        if activation == 'relu':
            layers.append(nn.ReLU())
        elif activation == 'sigmoid':
            layers.append(nn.Sigmoid())
        elif activation == 'tanh':
            layers.append(nn.Tanh())
    layers.append(nn.Linear(hidden_size, 1))
    
    model = nn.Sequential(*layers)
    
    # Xavier init for fair comparison
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
    
    x = torch.randn(32, 2)
    y = model(x)
    loss = y.mean()
    loss.backward()
    
    grad_norms = []
    for m in model.modules():
        if isinstance(m, nn.Linear) and m.weight.grad is not None:
            grad_norms.append(m.weight.grad.norm().item())
    
    return grad_norms

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

config = [
    ('sigmoid', '#e74c3c', 'Sigmoid — Vanishing Gradients'),
    ('tanh',    '#f39c12', 'Tanh — Mild Vanishing'),
    ('relu',    '#2ecc71', 'ReLU — Healthy Gradient Flow'),
]

for activation, color, label in config:
    norms = measure_gradient_flow(activation, num_layers=20)
    axes[0].plot(range(1, len(norms)+1), norms, color=color, linewidth=2.5, label=label, marker='o', markersize=4)

axes[0].set_xlabel('Layer (1=closest to output)', fontsize=12)
axes[0].set_ylabel('Gradient Norm', fontsize=12)
axes[0].set_title('Gradient Norms Across Layers\n(20-layer network, Xavier init)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_yscale('log')

# Show the "dead ReLU" problem — with bad initialization
torch.manual_seed(42)
x = torch.randn(100, 50)
# Large positive bias → all units active
# Large negative bias → dead ReLUs
for bias_val, color, label in [(-2.0, '#e74c3c', 'Bias=-2 (Dead ReLUs)'), 
                                 (0.0, '#3498db', 'Bias=0 (Normal)'),
                                 (1.0, '#2ecc71', 'Bias=+1 (All Active)')]:
    activations_per_layer = []
    a = x
    for _ in range(15):
        w = torch.randn(a.shape[1], 50) * 0.5
        b = torch.full((50,), bias_val)
        a = torch.relu(a @ w + b)
        frac_alive = (a > 0).float().mean().item()
        activations_per_layer.append(frac_alive)
    axes[1].plot(activations_per_layer, color=color, linewidth=2.5, label=label, marker='s', markersize=5)

axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='50% active (ideal)')
axes[1].set_xlabel('Layer depth', fontsize=12)
axes[1].set_ylabel('Fraction of Active Neurons', fontsize=12)
axes[1].set_title('Dead ReLU Problem\n(fraction of neurons with non-zero output)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()

## Part 3: Batch Normalization — Why It Works

BatchNorm normalizes each layer's pre-activations to have zero mean and unit variance, then applies learnable scale/shift:
$$\hat{x} = \frac{x - \mu_{\text{batch}}}{\sqrt{\sigma^2_{\text{batch}} + \epsilon}} \quad\quad y = \gamma \hat{x} + \beta$$

This prevents internal covariate shift and allows much higher learning rates.

In [ ]:
def build_deep_mlp(use_batchnorm=False, depth=10, hidden=128):
    layers = [nn.Linear(784, hidden)]
    for _ in range(depth - 1):
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(hidden))
        layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden, hidden))
    if use_batchnorm:
        layers.append(nn.BatchNorm1d(hidden))
    layers.append(nn.ReLU())
    layers.append(nn.Linear(hidden, 10))
    return nn.Sequential(*layers)

import torchvision
import torchvision.transforms as transforms

mnist_train = torchvision.datasets.MNIST('./data', train=True, download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]))
mnist_test  = torchvision.datasets.MNIST('./data', train=False, download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]))

train_loader = torch.utils.data.DataLoader(mnist_train, batch_size=512, shuffle=True)
test_loader  = torch.utils.data.DataLoader(mnist_test,  batch_size=512)

def train_and_track(model, epochs=5, lr=0.01, label=''):
    device = torch.device('cpu')
    model = model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    train_acc, test_acc = [], []
    
    for epoch in range(epochs):
        model.train()
        correct = total = 0
        for X, y in train_loader:
            X = X.view(X.size(0), -1).to(device)
            y = y.to(device)
            opt.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            opt.step()
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
        train_acc.append(correct / total)
        
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X, y in test_loader:
                X = X.view(X.size(0), -1)
                out = model(X)
                correct += (out.argmax(1) == y).sum().item()
                total += y.size(0)
        test_acc.append(correct / total)
        print(f'[{label}] Epoch {epoch+1} | Train: {train_acc[-1]:.1%} | Test: {test_acc[-1]:.1%}')
    return train_acc, test_acc

print('Training WITHOUT BatchNorm (10 layers)...')
no_bn_train, no_bn_test = train_and_track(build_deep_mlp(False), label='No BN')

print('\nTraining WITH BatchNorm (10 layers)...')
bn_train, bn_test = train_and_track(build_deep_mlp(True), label='With BN')

epochs = list(range(1, 6))
plt.figure(figsize=(10, 5))
plt.plot(epochs, [a*100 for a in no_bn_test], 'r-o', linewidth=2.5, label='No BatchNorm')
plt.plot(epochs, [a*100 for a in bn_test], 'g-o', linewidth=2.5, label='With BatchNorm')
plt.xlabel('Epoch'); plt.ylabel('Test Accuracy (%)')
plt.title('Effect of Batch Normalization on 10-Layer MLP', fontsize=13, fontweight='bold')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## Exercises

### Exercise A — Complete the Autograd Engine
Extend the `Value` class to support:
1. `sigmoid()` operation with correct gradient
2. `__truediv__` using the quotient rule: $\frac{d}{dx}\frac{f}{g} = \frac{f'g - fg'}{g^2}$
3. Verify each operation numerically using finite differences: $\frac{\partial L}{\partial x} \approx \frac{L(x+h) - L(x-h)}{2h}$ with $h=10^{-5}$

### Exercise B — Gradient Clipping
Exploding gradients can be fixed by clipping. Implement and demonstrate:
```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```
- Train a deep RNN on random sequences
- Plot gradient norm with and without clipping
- Show how clipping stabilizes training

### Exercise C — Skip Connections (ResNet)
The ResNet innovation: add a skip connection $F(x) + x$. Build a `ResBlock` and show it solves the degradation problem:
```python
class ResBlock(nn.Module):
    def forward(self, x):
        return self.layers(x) + x  # The key: gradient highway!
```
Compare gradient norms in a 20-layer plain network vs 20-layer ResNet.

### Discussion Questions
1. In the `Value` autograd, why do we accumulate gradients with `+=` instead of `=`?
2. Why does batch normalization allow higher learning rates? What would happen without it?
3. ResNet skip connections are a "gradient highway." Draw the computational graph of a ResBlock and trace the two gradient paths.